# 16 — Multimodal Agents — Reasoning Across Speech, Vision & Text

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/16_multimodal_agents_across_speech_and_vision.ipynb)

⏱ **Reading time:** ~35 min &nbsp;|&nbsp; **Prerequisites:** Notebooks 1–15, `agents/03_tool_use_and_function_calling.ipynb`

## Learning Objectives

By the end of this notebook you will be able to:

* **Explain** the multimodal-agent paradigm — an LLM orchestrator that dynamically selects modality-specific tools.
* **Design** tool schemas for vision, speech, and text-search capabilities with clear interfaces.
* **Implement** a ReAct-style agent loop that reasons over heterogeneous tool outputs.
* **Handle** errors gracefully when individual modality tools fail or return low-confidence results.
* **Profile** latency and cost across modalities to make informed architectural trade-offs.
* **Decide** when a flexible agent architecture is preferable to a fixed multimodal pipeline.

## 1 — The Multimodal Agent Concept

Traditional AI systems are specialists. A vision model classifies images; a speech model transcribes audio; a retrieval system fetches documents. Each lives in its own silo, and the developer must wire them together with hand-written glue code.

A **multimodal agent** takes a different approach. At its core sits a large language model that acts as an *orchestrator*. It receives a user task in natural language, **reasons** about which modality tools to invoke, **calls** those tools, **observes** the results, and decides whether additional steps are needed before producing a final answer. The key insight: the LLM does not need to *be* a vision model or an audio model — it just needs to know *when* and *how* to call one.

Consider three requests:

| Request | Modalities |
|---|---|
| *"What's happening in this security camera feed?"* | Vision |
| *"Summarize this meeting recording."* | Audio → Text |
| *"Compare the chart in this PDF with what the speaker said."* | Vision + Audio + Text |

A single-modality pipeline needs three separate code paths. A multimodal agent handles all three with the **same loop** — only tool selection changes.

The trade-off is complexity: every orchestration step adds latency, and the LLM can make mistakes in tool selection. Understanding this tension — flexibility vs. reliability — is the central theme of this notebook.

## 2 — Agent Architecture

The architecture follows the **ReAct** pattern (Reason + Act). The orchestrator alternates between *thinking* steps and *action* steps, grounding each decision in tool observations.

```
┌────────────┐
│  User Task  │
└─────┬──────┘
      │
      ▼
┌──────────────────┐
│  LLM Orchestrator │◄──────────────────────────┐
│  (Qwen3-8B 4-bit) │                            │
└─────┬────────────┘                             │
      │  Reason: which tool?                     │
      ▼                                          │
┌──────────────┐                                 │
│ Tool Selection │                                │
└──┬───┬───┬───┘                                 │
   │   │   │                                     │
   ▼   ▼   ▼                                     │
┌─────┐ ┌──────────┐ ┌─────────────┐            │
│Image│ │Transcribe│ │Search Docs  │            │
│Desc.│ │  Audio   │ │(Embeddings) │            │
└──┬──┘ └────┬─────┘ └──────┬──────┘            │
   │         │              │                    │
   └─────────┴──────────────┘                    │
              │                                  │
              ▼                                  │
     ┌────────────────┐    Need more tools?      │
     │ Combine Results │───── Yes ────────────────┘
     └───────┬────────┘
             │ No
             ▼
     ┌──────────────┐
     │ Final Answer  │
     └──────────────┘
```

**Key design decisions:**

1. **Tool schemas** — Each tool exposes a JSON schema (name, description, parameter types, return type) so the LLM can select tools without ambiguity.
2. **Message passing** — Tool outputs are serialized as text and appended to the conversation, giving the LLM full context for its next step.
3. **Stopping conditions** — The loop terminates when the LLM emits a *final answer* token or a maximum iteration count is reached.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────
!pip install -q transformers torch torchaudio datasets librosa bitsandbytes accelerate Pillow soundfile

import json, time, textwrap, warnings, os
from pathlib import Path
from typing import Any, Callable

import torch
import numpy as np
from PIL import Image
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig,
)

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 3 — Designing Multimodal Tools

A well-designed tool has four properties:

| Property | Why It Matters |
|---|---|
| **Clear schema** | The LLM selects tools by name + description; vague names cause mis-routing. |
| **Single modality** | A tool that handles images *and* audio is hard to test and reason about. |
| **Structured text output** | The LLM reads results as text; raw tensors are useless. |
| **Graceful failure** | `{"error": "..."}` is far better than crashing the loop. |

**Granularity trade-offs.** An overly coarse `analyze_everything(file)` gives the LLM no control. Overly fine tools (separate `detect_edges`, `detect_faces`, `detect_text`) force many sequential calls, multiplying latency. The sweet spot is **one tool per modality** with optional detail parameters.

In [ ]:
# ── Tool Schema Definitions ──────────────────────────────────────

TOOL_SCHEMAS = {
    "describe_image": {
        "name": "describe_image",
        "description": "Generate a natural-language description of an image.",
        "parameters": {
            "type": "object",
            "properties": {
                "image_path": {
                    "type": "string",
                    "description": "Path to the image file.",
                }
            },
            "required": ["image_path"],
        },
        "return_type": "string",
    },
    "transcribe_audio": {
        "name": "transcribe_audio",
        "description": "Transcribe speech in an audio file to text.",
        "parameters": {
            "type": "object",
            "properties": {
                "audio_path": {
                    "type": "string",
                    "description": "Path to the audio file.",
                }
            },
            "required": ["audio_path"],
        },
        "return_type": "string",
    },
    "search_documents": {
        "name": "search_documents",
        "description": "Semantic search over a document corpus. Returns top-k passages.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Natural-language search query.",
                },
                "top_k": {
                    "type": "integer",
                    "description": "Number of results to return.",
                    "default": 3,
                },
            },
            "required": ["query"],
        },
        "return_type": "list[string]",
    },
}

print(f"Registered {len(TOOL_SCHEMAS)} tool schemas:")
for name, schema in TOOL_SCHEMAS.items():
    print(f"  • {name}: {schema['description']}")

In [ ]:
# ── Image Description Tool ────────────────────────────────────────

image_captioner = pipeline(
    "image-to-text",
    model="Salesforce/blip-image-captioning-base",
    device=0 if DEVICE == "cuda" else -1,
)


def describe_image(image_path: str) -> dict:
    """Generate a natural-language description of an image."""
    try:
        img = Image.open(image_path).convert("RGB")
    except Exception as e:
        return {"error": f"Could not open image: {e}"}

    t0 = time.time()
    result = image_captioner(img, max_new_tokens=60)
    elapsed = time.time() - t0

    caption = result[0]["generated_text"]
    return {"caption": caption, "latency_s": round(elapsed, 3)}

In [ ]:
# Quick smoke test with a synthetic image
test_img = Image.new("RGB", (224, 224), color=(70, 130, 180))
test_img.save("_test_img.png")
print(describe_image("_test_img.png"))

In [ ]:
# ── Audio Transcription Tool ──────────────────────────────────────

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=0 if DEVICE == "cuda" else -1,
)


def transcribe_audio(audio_path: str) -> dict:
    """Transcribe speech in an audio file to text."""
    import librosa

    try:
        waveform, sr = librosa.load(audio_path, sr=16_000)
    except Exception as e:
        return {"error": f"Could not load audio: {e}"}

    if np.max(np.abs(waveform)) < 1e-4:
        return {"error": "Audio appears to be silent.", "transcript": ""}

    t0 = time.time()
    result = asr_pipe(
        {"raw": waveform, "sampling_rate": 16_000},
        return_timestamps=True,
    )
    elapsed = time.time() - t0

    return {
        "transcript": result["text"].strip(),
        "latency_s": round(elapsed, 3),
        "duration_s": round(len(waveform) / 16_000, 2),
    }

In [ ]:
# Smoke test with a 1-second sine wave
import soundfile as sf

sr = 16_000
tone = 0.5 * np.sin(2 * np.pi * 440 * np.arange(sr) / sr).astype(np.float32)
sf.write("_test_audio.wav", tone, sr)
print(transcribe_audio("_test_audio.wav"))

In [ ]:
# ── Document Search Tool ──────────────────────────────────────────
from transformers import AutoModel
import torch.nn.functional as F

CORPUS = [
    "Multimodal agents combine vision, speech, and text modalities.",
    "The ReAct pattern alternates between reasoning and acting.",
    "Whisper is an automatic speech recognition model by OpenAI.",
    "BLIP generates image captions from visual inputs.",
    "FAISS enables fast approximate nearest-neighbor search.",
    "Tool-use allows LLMs to call external functions.",
    "Quantization reduces model memory with minimal quality loss.",
    "Transformers use self-attention to process sequential data.",
]

embed_tok = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)
embed_model = AutoModel.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
).to(DEVICE)


def _embed(texts: list[str]) -> torch.Tensor:
    tokens = embed_tok(
        texts, padding=True, truncation=True, return_tensors="pt"
    ).to(DEVICE)
    with torch.no_grad():
        out = embed_model(**tokens)
    embs = out.last_hidden_state[:, 0]
    return F.normalize(embs, dim=-1)


corpus_embs = _embed(CORPUS)


def search_documents(query: str, top_k: int = 3) -> dict:
    """Semantic search over a document corpus. Returns top-k passages."""
    t0 = time.time()
    q_emb = _embed([query])
    scores = (q_emb @ corpus_embs.T).squeeze(0)
    idxs = scores.argsort(descending=True)[:top_k].tolist()
    elapsed = time.time() - t0
    return {
        "results": [
            {"text": CORPUS[i], "score": round(scores[i].item(), 4)}
            for i in idxs
        ],
        "latency_s": round(elapsed, 4),
    }


print(search_documents("How does speech recognition work?"))

## 4 — The Agent Loop

Our agent follows a **ReAct-style** loop:

1. **Observe** — Receive the user task (and any prior tool results).
2. **Think** — The LLM reasons about which tool to call next and why.
3. **Act** — Execute the chosen tool with the planned arguments.
4. **Observe** — Append the tool's structured output to the conversation.
5. **Think** — Decide whether another tool call is needed or the answer is complete.
6. **Answer** — Emit a final response that synthesizes all gathered evidence.

**Stopping conditions** prevent infinite loops:

* **Max iterations** — A hard cap (e.g., 5) on tool calls.
* **LLM signal** — The model outputs `FINAL_ANSWER:`, indicating it has enough information.

This keeps the agent both flexible and bounded.

In [ ]:
# ── Load Orchestrator LLM (Qwen3-8B, 4-bit) ─────────────────────

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

LLM_NAME = "Qwen/Qwen3-8B"
llm_tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    quantization_config=bnb_cfg,
    device_map="auto",
    trust_remote_code=True,
)
print(f"Loaded {LLM_NAME} in 4-bit on {llm_model.device}")

In [ ]:
# ── Tool Registry ────────────────────────────────────────────────

TOOL_FUNCTIONS = {
    "describe_image": describe_image,
    "transcribe_audio": transcribe_audio,
    "search_documents": search_documents,
}

SYSTEM_PROMPT = textwrap.dedent("""\
    You are a multimodal assistant with access to these tools:
    {tool_descriptions}

    To use a tool, respond EXACTLY in this format:
    TOOL_CALL: <tool_name> | <json_args>

    When you have enough information, respond with:
    FINAL_ANSWER: <your answer>

    Think step-by-step. Call only the tools you need.
    If the task mentions an image path, audio path, or document lookup, your first response should be a TOOL_CALL.

    Example trace:
    User: What objects are visible in the image at photo.png?
    Assistant: TOOL_CALL: describe_image | {\"image_path\": \"photo.png\"}
    User: Tool result: {\"caption\": \"a red car parked on a street\"}
    Assistant: FINAL_ANSWER: The image shows a red car parked on a street.
""")
print("System prompt and tool registry ready.")

In [ ]:
# ── MultimodalAgent Class ─────────────────────────────────────────

class MultimodalAgent:
    """ReAct-style agent that orchestrates multimodal tools."""

    def __init__(self, model, tokenizer, tools: dict, schemas: dict,
                 max_iterations: int = 5):
        self.model = model
        self.tokenizer = tokenizer
        self.tools = tools
        self.schemas = schemas
        self.max_iterations = max_iterations
        self.history: list[dict] = []

    def plan(self, task: str) -> str:
        """Ask the LLM what to do next."""
        tool_desc = "\n".join(
            f"- {s['name']}: {s['description']}"
            for s in self.schemas.values()
        )
        system = SYSTEM_PROMPT.format(tool_descriptions=tool_desc)
        messages = [{"role": "system", "content": system}]
        messages += self.history
        messages.append({"role": "user", "content": task})

        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(
            text, return_tensors="pt"
        ).to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(
                **inputs, max_new_tokens=256, do_sample=False,
            )
        return self.tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        ).strip()

    def execute_tool(self, tool_name: str, args: dict) -> dict:
        """Call the named tool with the given arguments."""
        if tool_name not in self.tools:
            return {"error": f"Unknown tool: {tool_name}"}
        try:
            return self.tools[tool_name](**args)
        except Exception as e:
            return {"error": str(e)}

    def synthesize(self, tool_outputs: list[dict]) -> str:
        """Ask the LLM to combine all tool results."""
        summary = "Tool results collected:\n"
        for i, out in enumerate(tool_outputs, 1):
            summary += f"  [{i}] {json.dumps(out, default=str)[:300]}\n"
        summary += "\nProvide a FINAL_ANSWER that synthesizes the above."
        return self.plan(summary)

    def run(self, task: str, verbose: bool = True) -> str:
        """Orchestrate the full ReAct loop."""
        self.history = []
        tool_outputs = []

        for step in range(self.max_iterations):
            if verbose:
                print(f"\n--- Step {step + 1} ---")
            response = self.plan(task)
            if verbose:
                print(f"LLM: {response[:200]}")

            if "FINAL_ANSWER:" in response:
                if not tool_outputs:
                    if verbose:
                        print("LLM tried to answer before using a tool; requesting a tool call.")
                    self.history.append(
                        {"role": "assistant", "content": response}
                    )
                    self.history.append(
                        {"role": "user",
                         "content": "Do not answer yet. First call the most relevant tool using TOOL_CALL: <tool_name> | <json_args>."}
                    )
                    continue
                return response.split("FINAL_ANSWER:", 1)[1].strip()

            if "TOOL_CALL:" in response:
                try:
                    call_str = response.split("TOOL_CALL:", 1)[1].strip()
                    parts = call_str.split("|", 1)
                    tool_name = parts[0].strip()
                    args = json.loads(parts[1].strip()) if len(parts) > 1 else {}
                except (IndexError, json.JSONDecodeError) as e:
                    self.history.append(
                        {"role": "assistant", "content": response}
                    )
                    self.history.append(
                        {"role": "user",
                         "content": f"Invalid tool call format: {e}. Use TOOL_CALL: <tool_name> | <json_args>."}
                    )
                    continue
                if verbose:
                    print(f"Calling: {tool_name}({args})")

                result = self.execute_tool(tool_name, args)
                tool_outputs.append({"tool": tool_name, **result})
                self.history.append(
                    {"role": "assistant", "content": response}
                )
                self.history.append(
                    {"role": "user",
                     "content": f"Tool result: {json.dumps(result, default=str)}"}
                )
                task = "Continue based on the tool results above."
            else:
                self.history.append(
                    {"role": "assistant", "content": response}
                )
                self.history.append(
                    {"role": "user",
                     "content": "Use either TOOL_CALL: <tool_name> | <json_args> or FINAL_ANSWER: <your answer>."}
                )

        if tool_outputs:
            return self.synthesize(tool_outputs)

        return "Max iterations reached without a valid tool call or final answer."


agent = MultimodalAgent(
    llm_model, llm_tok, TOOL_FUNCTIONS, TOOL_SCHEMAS
)
print("MultimodalAgent initialized.")

In [ ]:
# ── Task 1: Audio → Summarize ─────────────────────────────────────
answer = agent.run(
    "Transcribe the audio at _test_audio.wav and summarize the key points."
)
print(f"\n✅ Final Answer: {answer}")

In [ ]:
# ── Task 2: Vision → Describe ────────────────────────────────────
answer = agent.run(
    "What objects are visible in the image at _test_img.png?"
)
print(f"\n✅ Final Answer: {answer}")

In [ ]:
# ── Task 3: Audio → Search ───────────────────────────────────────
answer = agent.run(
    "Find documents related to speech recognition models."
)
print(f"\n✅ Final Answer: {answer}")

## 5 — Error Handling in Multimodal Agents

In production, tools *will* fail. Images get corrupted in transit. Audio files may contain only silence. A search index may return zero relevant results. A robust agent must handle these gracefully.

**Strategies for resilience:**

* **Graceful degradation** — If the vision tool fails, answer using whatever other evidence is available.
* **Fallback responses** — Return `{"error": "..."}` rather than raising an exception, so the LLM can reason about the failure.
* **Retry with different parameters** — A blurry image might work if resized; noisy audio might work with a longer context window.
* **Reporting uncertainty** — The final answer should indicate which modalities contributed and which failed.

In [ ]:
# ── Tool-Level Error Handling ─────────────────────────────────────

# 1. Missing image
print("Missing image :", describe_image("nonexistent_file.png"))

# 2. Silent audio
silent = np.zeros(16_000, dtype=np.float32)
sf.write("_silent.wav", silent, 16_000)
print("Silent audio  :", transcribe_audio("_silent.wav"))

# 3. Obscure search query
print("Obscure query :", search_documents(
    "quantum entanglement in black holes", top_k=1
))

In [ ]:
# ── Agent-Level Error Recovery ────────────────────────────────────
print("--- Agent with bad input ---")
answer = agent.run("Describe the image at missing_photo.jpg.")
print(f"\nAgent answer: {answer}")

## 6 — Cost and Latency Considerations

Different modalities have dramatically different computational profiles:

| Modality | Typical Latency | GPU Memory | Cost | Accuracy Notes |
|---|---|---|---|---|
| **Text search** | 5–20 ms | ~200 MB | Low | High for in-domain |
| **Image captioning** | 100–500 ms | ~1 GB | Medium | Good for photos; weaker on diagrams |
| **Audio transcription** | 0.5–5 s | ~1 GB | Medium–High | Excellent for clean speech |
| **LLM orchestrator** | 1–3 s / turn | ~5 GB (4-bit) | Highest | Depends on prompt quality |

**Parallel vs. sequential.** If the agent needs both vision and audio tools with no data dependency, run them concurrently — a 2 s audio call overlapped with a 0.4 s image call takes only 2 s total.

**Budget allocation.** Limit LLM reasoning turns (the most expensive component) and let cheaper tools do the heavy lifting.

In [ ]:
# ── Latency Profiling ─────────────────────────────────────────────
import matplotlib.pyplot as plt

profile = {}

t0 = time.time()
describe_image("_test_img.png")
profile["describe_image"] = time.time() - t0

t0 = time.time()
transcribe_audio("_test_audio.wav")
profile["transcribe_audio"] = time.time() - t0

t0 = time.time()
search_documents("speech recognition")
profile["search_documents"] = time.time() - t0

print("Latency profile (seconds):")
for tool, t in sorted(profile.items(), key=lambda x: -x[1]):
    print(f"  {tool:25s} {t:.4f}s")

In [ ]:
# ── Latency Visualization ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 3))
tools_list = list(profile.keys())
times = [profile[t] for t in tools_list]
colors = ["#4c72b0", "#dd8452", "#55a868"]
ax.barh(tools_list, times, color=colors[: len(tools_list)])
ax.set_xlabel("Latency (seconds)")
ax.set_title("Per-Tool Latency Profile")
plt.tight_layout()
plt.show()

## 7 — When Multimodal Agents vs. Specialized Pipelines

An agent adds an **orchestration layer** between the user and the tools. Choosing the right architecture depends on the use case.

| Factor | Use an Agent | Use a Pipeline |
|---|---|---|
| **Task openness** | Open-ended; user may ask anything | Well-defined and repeatable |
| **Modality variability** | Changes per request | Same every time |
| **User interaction** | Clarification may be needed | Fully automated |
| **Latency budget** | Seconds acceptable | Milliseconds matter |
| **Error tolerance** | Some failures OK | Every step must succeed |

Many production systems use a **hybrid**: a lightweight router dispatches to pre-built pipelines for common cases, falling back to a full agent loop for novel queries.

## Exercises

### Exercise 1 — Add an `extract_table` Tool

Create a new tool that extracts tabular data from an image. Register it in `TOOL_SCHEMAS` and `TOOL_FUNCTIONS`, then test the agent on *"What data is in the table in this image?"*

### Exercise 2 — Parallel Tool Execution

Modify `MultimodalAgent.run` so that when the LLM requests multiple tools in one step, they run concurrently using `concurrent.futures.ThreadPoolExecutor`.

### Exercise 3 — Confidence Scoring

Add a `confidence` field to the agent's final answer. Compute it as a weighted average of per-tool confidence scores and return a value between 0 and 1 alongside the text answer.

In [ ]:
# ── Exercise 1 Starter: extract_table tool ────────────────────────

def extract_table(image_path: str) -> dict:
    """Extract tabular data from an image."""
    try:
        img = Image.open(image_path)
    except Exception as e:
        return {"error": f"Could not open image: {e}"}

    # TODO: replace with real OCR (e.g., TrOCR, PaddleOCR)
    return {
        "rows": [["Header1", "Header2"], ["val1", "val2"]],
        "confidence": 0.0,
    }


# TODO: Register in TOOL_SCHEMAS and TOOL_FUNCTIONS

In [ ]:
# ── Exercise 2 Starter: Parallel tool execution ──────────────────
from concurrent.futures import ThreadPoolExecutor, as_completed


def execute_tools_parallel(
    tools_to_call: list[tuple[str, dict]],
    tool_functions: dict,
) -> list[dict]:
    """Execute multiple tool calls concurrently."""
    results = [None] * len(tools_to_call)
    with ThreadPoolExecutor(max_workers=len(tools_to_call)) as pool:
        futures = {}
        for idx, (name, args) in enumerate(tools_to_call):
            fn = tool_functions[name]
            futures[pool.submit(fn, **args)] = idx
        for future in as_completed(futures):
            idx = futures[future]
            try:
                results[idx] = future.result()
            except Exception as e:
                results[idx] = {"error": str(e)}
    return results


# Quick test: image + search in parallel
parallel_results = execute_tools_parallel(
    [("describe_image", {"image_path": "_test_img.png"}),
     ("search_documents", {"query": "vision models"})],
    TOOL_FUNCTIONS,
)
for r in parallel_results:
    print(r)

In [ ]:
# ── Exercise 3 Starter: Confidence scoring ───────────────────────

def compute_agent_confidence(tool_outputs: list[dict]) -> float:
    """Aggregate confidence from tool outputs (0.0–1.0)."""
    scores = []
    for out in tool_outputs:
        if "error" in out:
            scores.append(0.0)
        elif "score" in out:
            scores.append(float(out["score"]))
        elif "confidence" in out:
            scores.append(float(out["confidence"]))
        else:
            scores.append(0.5)
    return round(sum(scores) / max(len(scores), 1), 4)


demo_outputs = [
    {"caption": "a blue square", "latency_s": 0.2},
    {"error": "Audio appears to be silent."},
    {"results": [{"text": "...", "score": 0.87}], "score": 0.87},
]
print(f"Aggregate confidence: {compute_agent_confidence(demo_outputs)}")

# TODO: Integrate into MultimodalAgent.run

## Key Takeaways

1. **The orchestrator pattern** — A single LLM can coordinate vision, speech, and text tools without being natively multimodal; it just needs well-defined tool schemas.
2. **Tool design matters** — One tool per modality with structured JSON output gives the LLM the clearest reasoning signal.
3. **ReAct loops are powerful but bounded** — Alternating reason/act steps let the agent adapt; always enforce a max iteration count.
4. **Errors are normal** — Graceful degradation (`{"error": ...}` instead of crashing) is non-negotiable in multimodal systems.
5. **Profile before optimizing** — Measure tool latencies first; the bottleneck is almost always the LLM turns.
6. **Agents vs. pipelines is a spectrum** — Use agents for flexibility, pipelines for speed, and hybrid routers for both.

## References

* Yao, S. et al. (2022). [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629). *arXiv 2210.03629*.
* Schick, T. et al. (2023). [Toolformer: Language Models Can Teach Themselves to Use Tools](https://arxiv.org/abs/2302.04761). *arXiv 2302.04761*.
* Radford, A. et al. (2022). [Robust Speech Recognition via Large-Scale Weak Supervision](https://arxiv.org/abs/2212.04356) (Whisper). *arXiv 2212.04356*.
* Li, J. et al. (2022). [BLIP: Bootstrapping Language-Image Pre-training](https://arxiv.org/abs/2201.12086). *arXiv 2201.12086*.
* HuggingFace. [Transformers Agents Documentation](https://huggingface.co/docs/transformers/agents).

In [ ]:
# ── Cleanup temporary files ───────────────────────────────────────
for f in ["_test_img.png", "_test_audio.wav", "_silent.wav"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

---

**← Previous:** [15 — Speech + Document Workflows](15_speech_plus_document_workflows.ipynb) · **Next:** [17 — Video Understanding, Frame Sampling & Temporal Reasoning](17_video_understanding_frame_sampling_and_temporal_reasoning.ipynb) →

**Cross-reference:** The tool-calling patterns in this notebook build directly on [`agents/03_tool_use_and_function_calling.ipynb`](../agents/03_tool_use_and_function_calling.ipynb), which covers single-modality tool use in depth.